In [41]:
import os
import glob
import pandas as pd
import numpy as np
import datetime
from itertools import product
import math
from scipy.signal import butter, filtfilt
from sklearn.cluster import KMeans, DBSCAN
from sklearn.preprocessing import StandardScaler

In [42]:
def load_config(directory):
    file_name=[f for f in glob.glob(directory + "/*.RNH")]
    f = open(file_name[0],'r')
    conf = f.read()    
    t = {}
    conf = conf.split("\n")
    for i in conf:
        if i != "":
            temp = i.split(",")
            t[temp[0]] = temp[1].strip()
    return t

In [43]:
def parse_date(date):
    return datetime.datetime.strptime(date,'%Y/%m/%d %H:%M:%S')

In [44]:
one_hr = 36000
half_day = one_hr * 12
day = one_hr * 24
week = 7 * day

days_dict = {"half day":half_day, "day":day, "week":week}

In [45]:
def cutoff(r, data):
    r = (100 - r) / 1000
    b, a = butter(4, r)
    filtered = filtfilt(b, a, data)
    return filtered

In [46]:
def data_prepare(d, file_dir):
    conf = load_config(file_dir)

    file_names = [f for f in glob.glob(file_dir + "/*.RND")]

    open_files = [open(f,'r').read().replace(' ',"").replace(",","").split('\n') for f in file_names]

    data_concat = [x for lst in open_files for x in lst]
    data_cleaned = ' '.join(data_concat).split()
    sound_lists = [float(data.replace('O','0')) for data in data_cleaned]

    start = parse_date(conf['Start Time'])
    time = [start + datetime.timedelta(milliseconds=(100*i)) for i in range(len(sound_lists))]
    df = pd.DataFrame({'Period start':time, 'Leq':sound_lists})

    df_copy = df.copy()
    df_copy = df_copy[:days_dict[d]+1]

    sound_cutoff = cutoff(70, df_copy['Leq'])
    df_copy = df_copy.assign(Leq_filtered=sound_cutoff)

    return df_copy

In [47]:
def standardize(data):
    sound = np.array(data)
    sound = sound.reshape(-1,1)

    scaler = StandardScaler()
    sound_normalized = scaler.fit_transform(sound)

    return sound_normalized

In [48]:
def find_peaks(data, time, eps):
    data_copy = data.copy()
    X = time.reshape(-1, 1)
    dbscan = DBSCAN(eps=eps, min_samples=10).fit(X)

    data_copy = data_copy.iloc[time]
    data_copy['peak_group'] = dbscan.labels_
    data_copy = data_copy[data_copy['peak_group'] != -1]
    return data_copy['peak_group']

In [49]:
def merge_data(df1, df2):
    df_merge = pd.concat([df1, df2], axis=1)
    df_merge['peak_group'] = df_merge['peak_group'].fillna(-1)
    return df_merge

In [50]:
def computeL_eq_t(data):
    sum_pow = np.sum(np.power(10, data/10))
    x = 1/len(data) * sum_pow
    return 10 * math.log(x,10)

def SoundAddition(data):
    x = np.sum(np.power(10, data/10))
    return 10 * math.log(x,10)

def computeSEL(data):
    return computeL_eq_t(data) + 10 * math.log((len(data)*100)/1000)

In [51]:
def IQR_outlier(data, d_type):
    Q1 = np.percentile(data[d_type], 25)
    Q3 = np.percentile(data[d_type], 75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    df_result_cleaned = data[~((data[d_type] < lower_bound) | (data[d_type] > upper_bound))]
    return df_result_cleaned

In [52]:
def find_localpeak_alternative(data, eps):
    avg_cluster = [data[data['cluster'] == i]['Leq_filtered'].mean() for i in range(len(data['cluster'].unique()))]
    max_cluster = np.array(avg_cluster).argmax()
    time_on_max_clusters = np.array(data[data['cluster'] == max_cluster].index)
    
    peaks = find_peaks(data, time_on_max_clusters, eps)
    df_result = merge_data(data, peaks)

    peak_median = [np.median(df_result[df_result['peak_group'] == i].index) for i in peaks.unique() if (len(df_result[df_result['peak_group'] == i]) != 0)]
    unix_time = np.array(df_result['Period start'])
    results = []

    for c in peaks.unique():
        c_data = df_result[df_result['peak_group'] == c].index
        min_t, max_t = c_data.min(), c_data.max()
        peak_t = int(peak_median[c])

        in_which_hour = min_t//36000

        raw = np.array(df_result[df_result['peak_group'] == c]['Leq'])
        Leq = computeL_eq_t(raw)
        LAE = computeSEL(raw)

        r = {"hours":in_which_hour, "start_time": unix_time[min_t], "end_time": unix_time[max_t], "peak_time": peak_t,
             "interval": unix_time[max_t]-unix_time[min_t], "median":peak_median[c], "Leq":Leq, "Lae":LAE}

        results.append(r)

    df_result = pd.DataFrame(results)
    df_result['interval'] = df_result['interval'] / np.timedelta64(1, 's')

    df_result_cleaned = IQR_outlier(df_result, 'interval')

    return df_result_cleaned

In [53]:
def to_time(str_time):
	if (len(str_time) < 10) : return str_time
	else : return str_time[-9:]

def to_date(str_date):
	return str_date[:10]

def merge_time_date(str_time):
	if len(str_time.split(".")) > 1 :
		return datetime.datetime.strptime(str_time, '%Y-%m-%d %H:%M:%S.%f')
	else:
		return datetime.datetime.strptime(str_time, '%Y-%m-%d %H:%M:%S')

In [54]:
def find_accuracy(flights_file, peaks):
    lag_width = 30
    
    def match_cond(start,stop,date_time):
        return (date_time >= start-datetime.timedelta(milliseconds=lag_width*1000)) & (date_time <= stop+datetime.timedelta(milliseconds=lag_width*1000))
    
    peaks["start_time"] = peaks["start_time"].astype('str').apply(lambda a: merge_time_date(a))
    peaks["end_time"] = peaks["end_time"].astype('str').apply(lambda a: merge_time_date(a))

    min_peak, max_peak = min(peaks["start_time"]), max(peaks["end_time"])
    
    flights_file = flights_file[(flights_file['Datetime'] > min_peak) & (flights_file['Datetime'] < max_peak)]
    
    matched = [flights_file[match_cond(start,stop,flights_file['Datetime'])] for start, stop in zip(peaks['start_time'], peaks['end_time'])
                if len(flights_file[match_cond(start,stop,flights_file['Datetime'])]) > 0]
    
    return len(matched) / len(peaks)

In [55]:
def clean_flight(flights_file):
    flights_file_cleaned = flights_file.copy()

    flights_file_cleaned["Local Time"] = flights_file["Local Time"].apply(lambda a: to_time(a))
    flights_file_cleaned["Local Date"] = flights_file["Local Date"].apply(lambda a: to_date(a))
    
    flights_file_cleaned["Datetime"] = flights_file_cleaned["Local Date"] + " " + flights_file_cleaned["Local Time"]
    flights_file_cleaned["Datetime"] = flights_file_cleaned["Datetime"].apply(lambda a: merge_time_date(a))

    return flights_file_cleaned

In [57]:
flights = pd.read_excel("../sound_proj_2/data/Raw-data/full_flights.xlsx", dtype={'Local Date': str, 'Local Time': str})

flights_file = clean_flight(flights)

# สถานที่รอบๆ สนามบิน

In [ ]:
dir = {'Romruedee Romklao':"../sound_proj_2/data/Raw-data/1-Romruedee-Romklao-12-20-Mar-2014/AU1_0001/",
       'Central Village':"../sound_proj_2/data/Raw-data/2-Central-Village-3-11-Apr-2014/AU1_0002/",
       'Bangchalong Canal':"../sound_proj_2/data/Raw-data/3-Bangchalong-Canal-3-11-Apr-2014/AU1_0001/",
       'ThanaPlace':"../sound_proj_2/data/Raw-data/4-ThanaPlace-28-Apr-7-May-2014/AU1_0002/",
       'Romklao27':"../sound_proj_2/data/Raw-data/5-Romklao27-28-Apr-7-May-2014/AU1_0001/",
       'BangPla43':"../sound_proj_2/data/Raw-data/6-BangPla43-10-17-May-2014/AU1_0001/",
       'Bangsaothong':"../sound_proj_2/data/Raw-data/7-Bangsaothong-10-17-May-2014/AU1_0002/",
       'Habitia Ramindra':"../sound_proj_2/data/Raw-data/8-Habitia-Ramindra-20-26-May-2014/AU1_0001/"}

places = ['Romruedee Romklao', 'Central Village', 'Bangchalong Canal', 'ThanaPlace', 'Romklao27', 'BangPla43', 'Bangsaothong', 'Habitia Ramindra']
n_eps = [20, 30, 50, 100, 150, 200, 250, 300]

In [ ]:
df_dict = {p:data_prepare('week', dir[p]) for p in places}

In [19]:
def sound_cluster_k(data, k):
    data_copy = data.copy()
    data_copy['Leq_norm'] = standardize(data_copy['Leq_filtered'])
    sound_norm = np.array(data_copy['Leq_norm']).reshape(-1,1)

    x_kmeans = KMeans(n_clusters=k, random_state=0).fit(sound_norm)
    data_copy['cluster'] = x_kmeans.labels_
    return data_copy

In [20]:
n_clusters = [2,3,4]
df_kmeans = {f"({pl}, {k})":sound_cluster_k(df_dict[pl], k) for pl, k in product(places, n_clusters)}

In [33]:
data_list = [{'place':pl, 'kmeans_n_clusters':k, 'dbscan_epsilon':eps, 'accuracy score':find_accuracy(flights_file, find_localpeak_alternative(df_kmeans[f"({pl}, {k})"], eps))} for pl, k, eps in product(places, n_clusters, n_eps)]

In [34]:
df = pd.DataFrame(data_list)
df.to_csv("../sound_proj_2/data/Processed-data/output_viz_clustering.csv",index=False)

# Visualization

In [35]:
import plotly.express as px

In [36]:
df_viz = pd.read_csv('../sound_proj_2/data/Processed-data/output_viz_clustering.csv')
df_viz.head()

,place,kmeans_n_clusters,dbscan_epsilon,accuracy score
0,Romruedee Romklao,2,20,0.709818
1,Romruedee Romklao,2,30,0.710423
2,Romruedee Romklao,2,50,0.708352
3,Romruedee Romklao,2,100,0.707605
4,Romruedee Romklao,2,150,0.708772


In [37]:
df_viz_k_2 = df_viz[df_viz['kmeans_n_clusters'] == 2].sort_values('dbscan_epsilon')
df_viz_k_3 = df_viz[df_viz['kmeans_n_clusters'] == 3].sort_values('dbscan_epsilon')
df_viz_k_4 = df_viz[df_viz['kmeans_n_clusters'] == 4].sort_values('dbscan_epsilon')

In [38]:
fig = px.line(df_viz_k_2, x="dbscan_epsilon", y="accuracy score", color="place", symbol="place")
fig.update_layout(title='Accuracy score of Aircraft model when k=2',
                   xaxis_title='Epsilon',
                   yaxis_title='Accuracy score')

fig.show()

In [39]:
fig = px.line(df_viz_k_3, x="dbscan_epsilon", y="accuracy score", color="place", symbol="place")
fig.update_layout(title='Accuracy score of Aircraft model when k=3',
                   xaxis_title='Epsilon',
                   yaxis_title='Accuracy score')
fig.show()

In [40]:
fig = px.line(df_viz_k_4, x="dbscan_epsilon", y="accuracy score", color="place", symbol="place")
fig.update_layout(title='Accuracy score of Aircraft model when k=4',
                   xaxis_title='Epsilon',
                   yaxis_title='Accuracy score')
fig.show()